In [1]:
import pymysql
from typing import *

def get_db_connection(dbms_host_ip: str, 
                      dbms_username: str, 
                      dbms_password: str, 
                      db_name: str=None, 
                      port: int = 3306) -> pymysql.connections.Connection:
    """MySQL(MariaDB) 데이터베이스 연결 객체를 생성하고 반환합니다.

    주어진 연결 정보(IP, 계정, 포트 등)를 사용하여 데이터베이스와의 세션을 맺습니다.
    db_name을 지정하지 않으면 특정 데이터베이스에 종속되지 않은 전역 커넥션을 반환합니다.

    Args:
        dbms_host_ip (str): 접속할 데이터베이스 서버의 IP 주소 또는 호스트명.
        dbms_username (str): 데이터베이스 접속 계정명.
        dbms_password (str): 데이터베이스 접속 계정의 비밀번호.
        db_name (str, optional): 접속 후 사용할 기본 데이터베이스(스키마) 이름. 기본값은 None.
        port (int, optional): 데이터베이스 서버의 포트 번호. 기본값은 3306.

    Returns:
        pymysql.connections.Connection: 성공적으로 연결된 PyMySQL 커넥션 객체.
    """
    config = {
        "host": dbms_host_ip,
        "user": dbms_username,
        "password": dbms_password,
        "port": port,
        "charset": "utf8mb4"
    }
    
    if db_name is not None:
        config["database"] = db_name
        
    return pymysql.connect(**config)


def db_run_sql(con: pymysql.connections.Connection, 
               sql_list: List[str]) -> List[Any]:
    """여러 개의 SQL 문을 순차적으로 실행하고 결과를 반환합니다.

    입력받은 쿼리 리스트를 하나의 논리적 작업 단위(트랜잭션)로 묶어서 실행합니다.
    SELECT 쿼리는 조회된 데이터를 반환하고, DML(INSERT, UPDATE, DELETE) 쿼리는
    영향을 받은 행(row)의 개수를 반환합니다.

    Args:
        con (pymysql.connections.Connection): 활성화된 PyMySQL 커넥션 객체.
        sql_list (List[str]): 순차적으로 실행할 SQL 쿼리 문자열들이 담긴 리스트.

    Returns:
        List[Any]: 각 SQL 쿼리의 실행 결과가 담긴 리스트.
            - SELECT 결과: 딕셔너리 형태의 데이터 리스트 (예: [{"id": 1, "name": "A"}])
            - DML 결과: 영향을 받은 행 개수를 담은 딕셔너리 (예: {"affected_rows": 1})

    Raises:
        pymysql.MySQLError: SQL 실행 중 오류가 발생할 경우 트랜잭션을 롤백하고 예외를 상위로 던집니다.
    """
    results = []
    try:
        with con.cursor(pymysql.cursors.DictCursor) as cursor:
            for sql in sql_list:
                cursor.execute(sql)
                if cursor.description is not None:
                    results.append(cursor.fetchall())
                else:
                    results.append({"affected_rows": cursor.rowcount})
        con.commit()
    except pymysql.MySQLError as e:
        con.rollback()
        #오류처리 할 거면 하기
        raise e
    return results


def db_create_user(con: pymysql.connections.Connection, 
                   new_user_name: str, 
                   new_user_password: str, 
                   new_user_hostname: str='%') -> bool:
    """새로운 데이터베이스 유저(계정)를 생성합니다.

    호출 시 제공된 이름, 호스트, 비밀번호를 사용하여 글로벌 수준의 접속 계정을 만듭니다.
    내부적으로 db_run_sql을 사용하여 쿼리를 실행하며, 계정 중복 등의 예외 발생 시 
    프로그램이 중단되지 않고 False를 반환하도록 처리합니다.

    Args:
        con (pymysql.connections.Connection): 계정 생성 권한(root 등)이 있는 커넥션 객체.
        new_user_name (str): 새로 생성할 유저의 식별 이름.
        new_user_password (str): 새로 생성할 유저의 비밀번호.
        new_user_hostname (str, optional): 유저 접속을 허용할 호스트(IP). 기본값은 '%' (모든 IP 허용).

    Returns:
        bool: 유저 생성이 정상적으로 완료되면 True, 에러가 발생하여 실패하면 False.
    """
    sql = [
        f"CREATE USER '{new_user_name}'@'{new_user_hostname}' IDENTIFIED BY '{new_user_password}';"
    ]
    try:
        db_run_sql(con, sql)
    except pymysql.MySQLError as e:
        # 오류처리 할 거면 하기
        return False
    
    return True
    

def db_grant_user(con: pymysql.connections.Connection,
                  target_user_name: str,
                  target_host: str = '%',
                  privileges: str = 'ALL PRIVILEGES',
                  target_db: str = '*',
                  target_table: str = '*') -> bool:
    """특정 유저에게 데이터베이스 접근 및 조작 권한을 부여합니다.

    파라미터를 생략하면 대상 유저에게 서버 전체(*.*)에 대한 모든 권한(ALL PRIVILEGES)을 부여합니다.
    보안 정책에 따라 특정 데이터베이스, 테이블, 권한 종류를 세밀하게 지정하여 
    최소 권한의 원칙(Principle of Least Privilege)을 적용할 수 있습니다.
    실행 성공 시 FLUSH PRIVILEGES를 통해 메모리에 권한을 즉시 동기화합니다.

    Args:
        con (pymysql.connections.Connection): 권한 부여 권한이 있는 커넥션 객체.
        target_user_name (str): 권한을 부여받을 대상 유저의 이름.
        target_host (str, optional): 대상 유저의 접속 허용 호스트. 기본값은 '%'.
        privileges (str, optional): 부여할 권한 (예: 'SELECT', 'INSERT, UPDATE'). 기본값은 'ALL PRIVILEGES'.
        target_db (str, optional): 권한을 적용할 대상 데이터베이스. 기본값은 '*' (전체 DB).
        target_table (str, optional): 권한을 적용할 대상 테이블. 기본값은 '*' (전체 테이블).

    Returns:
        bool: 권한 부여가 정상적으로 완료되면 True, 문법 오류나 권한 부족 등으로 실패하면 False.
    """
    sql = [
        f"GRANT {privileges} ON {target_db}.{target_table} TO '{target_user_name}'@'{target_host}';",
        "FLUSH PRIVILEGES;"
    ]
    try:
        db_run_sql(con, sql)  
    except pymysql.MySQLError as e:
        # 오류처리 할 거면 하기
        return False 
    return True

def db_select_data(con: pymysql.connections.Connection,
                   table: str,
                   columns: str = "*",
                   where: str = None,
                   group_by: str = None,
                   having: str = None,
                   order_by: str = None,
                   limit: int = None) -> Optional[List[Dict[str, Any]]]:
    """SQL 인젝션 방어 없이, 순수 문자열 기반으로 동적 SELECT 쿼리를 조립하고 실행합니다.

    기존의 db_run_sql 함수를 수정 없이 재활용하기 위해 설계된 경량화 조회 함수입니다.
    파라미터 바인딩(%s)을 지원하지 않으므로, 외부 사용자 입력값을 그대로 사용할 경우 
    SQL 인젝션 공격에 매우 취약합니다. 따라서 신뢰할 수 있는 내부 시스템 쿼리나 
    관리자용 폐쇄망 환경에서 제한적으로 사용하는 것을 권장합니다.

    Args:
        con (pymysql.connections.Connection): 활성화된 PyMySQL 커넥션 객체.
        table (str): 조회할 대상 테이블 이름 (FROM 절).
        columns (str, optional): 가져올 컬럼 목록. 기본값은 "*".
        where (str, optional): 완성된 조건식 문자열 (예: "age >= 20 AND status = 'active'"). 기본값은 None.
        group_by (str, optional): 그룹화할 컬럼명. 기본값은 None.
        having (str, optional): 그룹화 이후의 조건식 문자열 (예: "COUNT(id) > 5"). 기본값은 None.
        order_by (str, optional): 정렬 기준 문자열 (예: "created_at DESC"). 기본값은 None.
        limit (int, optional): 가져올 최대 데이터 행(Row)의 개수. 기본값은 None.

    Returns:
        Optional[List[Dict[str, Any]]]: 조건에 맞는 데이터가 담긴 딕셔너리 리스트. 
            매칭되는 데이터가 없으면 빈 리스트([])를 반환하며, 
            문법 오류 등 데이터베이스 처리 중 예외가 발생하면 None을 반환합니다.
    """
    sql_parts = [f"SELECT {columns} FROM {table}"]
    if where: sql_parts.append(f"WHERE {where}")
    if group_by: sql_parts.append(f"GROUP BY {group_by}")
    if having: sql_parts.append(f"HAVING {having}")
    if order_by: sql_parts.append(f"ORDER BY {order_by}")
    if limit is not None: sql_parts.append(f"LIMIT {limit}")
    final_sql = " ".join(sql_parts)
    try:
        return db_run_sql(con, [final_sql])[0]
    except pymysql.MySQLError:
        # 오류처리 할 거면 하기
        return None
    

def db_get_databases(con: pymysql.connections.Connection) -> Optional[List[str]]:
    """DBMS 서버 내에 존재하는 모든 데이터베이스(스키마)의 이름 목록을 반환합니다.

    내부적으로 'SHOW DATABASES;' 쿼리를 실행하며, 결과를 파싱하여 
    데이터베이스 이름만 담긴 깔끔한 문자열 리스트로 반환합니다.

    Args:
        con (pymysql.connections.Connection): 활성화된 PyMySQL 커넥션 객체.

    Returns:
        List[str]: 데이터베이스 이름 리스트 (예: ['mysql', 'sys', 'my_database']), 오류 발생 시 None
    """
    sql = ["SHOW DATABASES;"]
    
    try:
        results = db_run_sql(con, sql)[0]
        return [list(row.values())[0] for row in results]
    except pymysql.MySQLError as e:
        # 오류처리 할 거면 하기
        return None


def db_get_tables(con: pymysql.connections.Connection, 
                  target_db: str = None) -> Optional[List[str]]:
    """특정 데이터베이스 안에 존재하는 모든 테이블의 이름 목록을 반환합니다.

    target_db를 지정하면 해당 DB의 테이블을 조회하고, 
    생략하면 현재 커넥션이 맺어진 기본 DB의 테이블 목록을 조회합니다.

    Args:
        con (pymysql.connections.Connection): 활성화된 PyMySQL 커넥션 객체.
        target_db (str, optional): 테이블을 조회할 대상 데이터베이스. 기본값은 None.

    Returns:
        List[str]: 테이블 이름 리스트 (예: ['users', 'orders']), 오류 발생 시 None
    """
    if target_db:
        sql = [f"SHOW TABLES FROM `{target_db}`;"]
    else:
        sql = ["SHOW TABLES;"]
    try:
        results = db_run_sql(con, sql)[0]
        return [list(row.values())[0] for row in results]
    except pymysql.MySQLError as e:
        # 오류처리 할 거면 하기
        return None

In [ ]:
def main():
    con = None
    print("🚀 MySQL/MariaDB 유틸리티 테스트를 시작합니다.")
    
    while True:
        print("\n" + "="*40)
        print("🛠️  데이터베이스 관리 메뉴")
        print("="*40)
        print("1. DB 연결 (가장 먼저 실행해야 합니다)")
        print("2. 전체 데이터베이스 목록 조회")
        print("3. 테이블 목록 조회 (현재 연결된 DB 기준)")
        print("4. 데이터 조회 (SELECT 테스트)")
        print("5. 새 유저 생성")
        print("6. 유저 권한 부여")
        print("7. 임의의 SQL 직접 실행")
        print("0. 프로그램 종료")
        print("="*40)
        
        choice = input("▶ 원하는 메뉴 번호를 입력하세요: ").strip()
        
        if choice == '0':
            print("프로그램을 종료합니다. 수고하셨습니다!")
            if con and con.open:
                con.close()
                print("[Info] DB 연결이 안전하게 해제되었습니다.")
            break
            
        elif choice == '1':
            print("\n[DB 연결 설정]")
            host = input("Host IP (기본값 127.0.0.1): ") or "127.0.0.1"
            user = input("Username (기본값 root): ") or "root"
            password = input("Password: ")
            db = input("Target DB명 (없으면 엔터): ") or None
            
            try:
                con = get_db_connection(host, user, password, db)
                print("✅ DB 연결에 성공했습니다!")
            except Exception as e:
                print(f"❌ 연결 실패: {e}")
                
        else:
            # 1번과 0번을 제외한 모든 메뉴는 커넥션이 필수입니다.
            if con is None or not con.open:
                print("⚠️ [경고] 먼저 1번 메뉴를 통해 DB에 연결해주세요.")
                continue
                
            if choice == '2':
                print("\n[데이터베이스 목록]")
                databases = db_get_databases(con)
                if databases is not None:
                    for i, db_name in enumerate(databases, 1):
                        print(f"  {i}. {db_name}")
                else:
                    print("❌ 데이터베이스 목록을 불러오지 못했습니다.")
                    
            elif choice == '3':
                target = input("\n조회할 DB명 (현재 연결된 DB를 보려면 엔터): ") or None
                tables = db_get_tables(con, target)
                if tables is not None:
                    print(f"\n[테이블 목록]")
                    if not tables:
                        print("  (테이블이 없습니다)")
                    for i, tb_name in enumerate(tables, 1):
                        print(f"  {i}. {tb_name}")
                else:
                    print("❌ 테이블 목록을 불러오지 못했습니다.")
                    
            elif choice == '4':
                print("\n[데이터 조회 테스트]")
                table_name = input("조회할 테이블명: ")
                if not table_name:
                    print("테이블명은 필수입니다.")
                    continue
                    
                limit_val = input("최대 조회 갯수 (기본값 5): ") or "5"
                where_cond = input("WHERE 조건 (예: id > 0) / 없으면 엔터: ") or None
                
                results = db_select_data(
                    con, 
                    table=table_name, 
                    where=where_cond, 
                    limit=int(limit_val)
                )
                
                if results is not None:
                    print(f"\n✅ 조회 결과 (총 {len(results)}건):")
                    for row in results:
                        print(f"  - {row}")
                else:
                    print("❌ 데이터 조회에 실패했습니다. (테이블명이나 조건을 확인하세요)")
                    
            elif choice == '5':
                print("\n[새 유저 생성]")
                new_user = input("생성할 유저 ID: ")
                new_pw = input("생성할 유저 비밀번호: ")
                
                if db_create_user(con, new_user, new_pw):
                    print(f"✅ '{new_user}' 유저가 성공적으로 생성되었습니다.")
                else:
                    print("❌ 유저 생성 실패. (이미 존재하는 유저인지 확인하세요)")
                    
            elif choice == '6':
                print("\n[유저 권한 부여]")
                target_user = input("권한을 부여할 유저 ID: ")
                target_db = input("권한을 부여할 대상 DB (기본값 *): ") or "*"
                priv = input("부여할 권한 (기본값 ALL PRIVILEGES): ") or "ALL PRIVILEGES"
                
                if db_grant_user(con, target_user, privileges=priv, target_db=target_db):
                    print(f"✅ '{target_user}' 유저에게 권한({priv}) 부여 완료!")
                else:
                    print("❌ 권한 부여 실패.")
                    
            elif choice == '7':
                print("\n[임의의 SQL 실행]")
                custom_sql = input("실행할 SQL 문을 입력하세요: ")
                try:
                    results = db_run_sql(con, [custom_sql])
                    print(f"✅ 실행 완료! 결과: {results}")
                except Exception as e:
                    print(f"❌ 실행 중 오류 발생: {e}")
                    
            else:
                print("⚠️ 올바른 메뉴 번호를 입력해주세요.")

main()

🚀 MySQL/MariaDB 유틸리티 테스트를 시작합니다.

🛠️  데이터베이스 관리 메뉴
1. DB 연결 (가장 먼저 실행해야 합니다)
2. 전체 데이터베이스 목록 조회
3. 테이블 목록 조회 (현재 연결된 DB 기준)
4. 데이터 조회 (SELECT 테스트)
5. 새 유저 생성
6. 유저 권한 부여
7. 임의의 SQL 직접 실행
0. 프로그램 종료


▶ 원하는 메뉴 번호를 입력하세요:  1



[DB 연결 설정]


Host IP (기본값 127.0.0.1):  172.16.1.2
Username (기본값 root):  root
Password:  asd123!@
Target DB명 (없으면 엔터):  joomla_db


✅ DB 연결에 성공했습니다!

🛠️  데이터베이스 관리 메뉴
1. DB 연결 (가장 먼저 실행해야 합니다)
2. 전체 데이터베이스 목록 조회
3. 테이블 목록 조회 (현재 연결된 DB 기준)
4. 데이터 조회 (SELECT 테스트)
5. 새 유저 생성
6. 유저 권한 부여
7. 임의의 SQL 직접 실행
0. 프로그램 종료


...


✅ 조회 결과 (총 1건):
  - {'id': 1, 'asset_id': 56, 'published': 1, 'title': 'COM_WORKFLOW_BASIC_WORKFLOW', 'description': '', 'extension': 'com_content.article', 'default': 1, 'ordering': 1, 'created': datetime.datetime(2026, 5, 26, 8, 41, 8), 'created_by': 773, 'modified': datetime.datetime(2026, 5, 26, 8, 41, 8), 'modified_by': 773, 'checked_out_time': None, 'checked_out': None}

🛠️  데이터베이스 관리 메뉴
1. DB 연결 (가장 먼저 실행해야 합니다)
2. 전체 데이터베이스 목록 조회
3. 테이블 목록 조회 (현재 연결된 DB 기준)
4. 데이터 조회 (SELECT 테스트)
5. 새 유저 생성
6. 유저 권한 부여
7. 임의의 SQL 직접 실행
0. 프로그램 종료


▶ 원하는 메뉴 번호를 입력하세요:  5



[새 유저 생성]


생성할 유저 ID:  testuset
생성할 유저 비밀번호:  asd123


✅ 'testuset' 유저가 성공적으로 생성되었습니다.

🛠️  데이터베이스 관리 메뉴
1. DB 연결 (가장 먼저 실행해야 합니다)
2. 전체 데이터베이스 목록 조회
3. 테이블 목록 조회 (현재 연결된 DB 기준)
4. 데이터 조회 (SELECT 테스트)
5. 새 유저 생성
6. 유저 권한 부여
7. 임의의 SQL 직접 실행
0. 프로그램 종료
